In [ ]:
import pandas as pd
from nli_project.features.vectorizer import NLITextVectorizer

In [2]:
df_train = pd.DataFrame({
    "premise": [
        ["el", "gato", "duerme", "en", "el", "sofa"],
        ["un", "perro", "corre", "por", "el", "parque"]
    ],
    "hypothesis": [
        ["hay", "un", "felino", "descansando"],
        ["el", "animal", "esta", "quieto"]
    ]
})


vectorizer = NLITextVectorizer(
    columns=("premise", "hypothesis"),
    method="tfidf",
    use_pos=False, # No uso etiquetas POS, solo tengo tokens
    sublinear_tf=True, # Aplico escala logarítmica de la frecuencia
    max_features=5000,
    word_ngram_range=(1, 1), # Analizo para palabras unigramas solo 
    min_df=1  # Descarto términos que aparecen en menos de 1 documento
              # Numero de documentos = numero de instancias (si partimos de 1 columna de tokens)
              # o el doble de instancias si partimos de 2 columnas (premise, hypothesis), ver fit()
)

X_train_vectors = vectorizer.fit_transform(df_train)

print("Tipo de salida:", type(X_train_vectors))
print("Forma (shape) de la matriz de vectores:", X_train_vectors.shape)

X_train_vectors.toarray()


# Shape (2,32) debido a la concatenacion horizontal de las matrices de TF-IDF de premisa e hipótesis,
# cada una con 16 características (o palabras únicas) en este ejemplo.
# Matriz de Premisas: 2 filas × 16 columnas
# Matriz de Hipótesis: 2 filas × 16 columnas
# Concatenación horizontal (16 + 16): 2 filas × 32 columna
# Ver método transform() de NLITextVectorizer

Tipo de salida: <class 'scipy.sparse._csr.csr_matrix'>
Forma (shape) de la matriz de vectores: (2, 32)


array([[0.        , 0.        , 0.        , 0.43988705, 0.47539207,
        0.43988705, 0.        , 0.        , 0.43988705, 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.43988705,
        0.        , 0.        , 0.        , 0.52547276, 0.        ,
        0.        , 0.        , 0.        , 0.52547276, 0.        ,
        0.52547276, 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.41428876],
       [0.        , 0.44592217, 0.        , 0.        , 0.28462633,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.44592217, 0.44592217, 0.44592217, 0.        , 0.        ,
        0.35157016, 0.5417361 , 0.        , 0.        , 0.        ,
        0.34578314, 0.        , 0.5417361 , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.5417361 ,
        0.        , 0.        ]], dtype=float32)

In [3]:
vectorizer = NLITextVectorizer(
    columns=("premise", "hypothesis"),
    method="word_char",
    use_pos=False, # No uso etiquetas POS, solo tengo tokens
    sublinear_tf=True, # Aplico escala logarítmica de la frecuencia
    max_features=5000,
    word_ngram_range=(1, 1), # Analizo unigramas solo para palabras
    char_ngram_range=(3, 5), # Analizo n-gramas de caracteres de longitud 3 a 5
    min_df=1  # Descarto términos que aparecen en menos de 1 documento
              # Numero de documentos = numero de instancias (si partimos de 1 columna de tokens)
              # o el doble de instancias si partimos de 2 columnas (premise, hypothesis), ver fit()
)

X_train_vectors = vectorizer.fit_transform(df_train)

print("Tipo de salida:", type(X_train_vectors))
print("Forma (shape) de la matriz de vectores:", X_train_vectors.shape)

X_train_vectors.toarray()

# A nivel de palabras (word): Extrae los unigramas de palabras. 
# Hay 16 palabras únicas en el corpus, este bloque aporta 16 columnas para premisa y 16 para hipotesis

#A nivel de caracteres (char): Extrae n-gramas de caracteres con el rango (3, 5) (secuencias de 3, 4 y 5 letras, incluyendo espacios y sufijos/raíces). 
# Al generar todas las combinaciones posibles de caracteres este vectorizador encuentra 254 características únicas en la premisa y 254 características únicas en la hipótesis.

# Total (al combinar horizontalmente): 16 + 16 + 254 + 254 = 540 columnas (características) en la matriz final de vectores.

Tipo de salida: <class 'scipy.sparse._csr.csr_matrix'>
Forma (shape) de la matriz de vectores: (2, 540)


array([[0.        , 0.        , 0.        , ..., 0.12380336, 0.12380336,
        0.12380336],
       [0.        , 0.44592217, 0.        , ..., 0.        , 0.        ,
        0.        ]], shape=(2, 540), dtype=float32)

In [4]:
df_1col_pos = pd.DataFrame({
    "tokens": [
        [("el", "DET"), ("gato", "NOUN"), ("duerme", "VERB"), ("entonces", "ADV"), ("hay", "VERB"), ("un", "DET"), ("felino", "NOUN")],
        [("un", "DET"), ("perro", "NOUN"), ("corre", "VERB"), ("por", "ADP"), ("el", "DET"), ("parque", "NOUN")]
    ]
})

vectorizer_1 = NLITextVectorizer(
    columns=("tokens",),
    method="embeddings",
    use_pos=True  # Activo POS tags
)

X_vecs_1 = vectorizer_1.fit_transform(df_1col_pos)

print("Forma (1 col, con POS):", X_vecs_1.shape) 
# Salida: (2, 384) -> Una sola matriz de embeddings de 384 dimensiones por instancia.
X_vecs_1 # No hace falta llamar a toarray() ya que la salida es un array de numpy, 
         # en vez de una matriz dispersa como en los tfidf o count vectorizer.

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 570.44it/s]


Forma (1 col, con POS): (2, 384)


In [5]:
df_2col_pos = pd.DataFrame({
    "premise": [
        [("el", "DET"), ("gato", "NOUN"), ("duerme", "VERB")],
        [("un", "DET"), ("perro", "NOUN"), ("corre", "VERB")]
    ],
    "hypothesis": [
        [("hay", "VERB"), ("un", "DET"), ("felino", "NOUN")],
        [("el", "animal", "NOUN"), ("esta", "AUX"), ("quieto", "ADJ")]
    ]
})

vectorizer_2 = NLITextVectorizer(
    columns=("premise", "hypothesis"),
    method="embeddings",
    use_pos=True
)

X_vecs_2 = vectorizer_2.fit_transform(df_2col_pos)

print("Forma (2 cols, con POS):", X_vecs_2.shape) 
# Aquí la salida es (2, 1536) porque el vectorizador genera embeddings de 384 dimensiones para cada columna (premise y hypothesis)
# Luego concatena horizontalmente los vectores de premisa, hipótesis, la diferencia absoluta y el producto elemento a elemento (muy poderoso para transformers)
# Resultado: 4 * 384 = 1536 características por instancia. 
# Salida: (2, 1536) -> 384 (premise) + 384 (hypothesis) + 384 (diferencia abs) + 384 (producto)
X_vecs_2

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 613.69it/s]


Forma (2 cols, con POS): (2, 1536)


array([[-1.3074461e-02,  3.3406179e-02, -5.9471643e-03, ...,
        -5.5743301e-05,  2.4317260e-04, -2.5384803e-04],
       [-3.7877338e-03,  4.9693786e-02, -4.9065221e-02, ...,
         3.3497654e-05,  3.7586762e-04,  5.4976642e-03]],
      shape=(2, 1536), dtype=float32)

In [6]:
df_1col_nopos = pd.DataFrame({
    "texto_completo": [
        [("el", "DET"), ("gato", "NOUN"), ("duerme", "VERB")],
        [("un", "DET"), ("perro", "NOUN"), ("corre", "VERB")]
    ]
})

vectorizer_3 = NLITextVectorizer(
    columns=("texto_completo",),
    method="embeddings",
    use_pos=False  # Ignora las etiquetas POS (mismo resultado que si se pasara una lista de strings sin tuplas)
                   # Se vectoriza del mismo modo 2 tokens idénticos pero con distinta categoría gramatical (ej: hunt verb y hunt noun)
)


X_vecs_3 = vectorizer_3.fit_transform(df_1col_nopos)

print("Forma (1 col, sin POS):", X_vecs_3.shape) 
# Salida: (2, 384) -> Devuelve únicamente el embedding limpio del texto completo.

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 711.72it/s]


Forma (1 col, sin POS): (2, 384)


In [7]:
df_2col_nopos = pd.DataFrame({
    "premise": [
        ["el", "gato", "duerme", "en", "el", "sofa"],
        ["un", "perro", "corre", "por", "el", "parque"]
    ],
    "hypothesis": [
        ["hay", "un", "felino", "descansando"],
        ["el", "animal", "esta", "quieto"]
    ]
})

vectorizer_4 = NLITextVectorizer(
    columns=("premise", "hypothesis"),
    method="embeddings",
    use_pos=False
)

X_vecs_4 = vectorizer_4.fit_transform(df_2col_nopos)

print("Forma (2 cols, sin POS):", X_vecs_4.shape) 
# Salida: (2, 1536) -> Concatena [premisa, hipótesis, |premisa - hipótesis|, premisa * hipótesis]
X_vecs_4

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 560.12it/s]


Forma (2 cols, sin POS): (2, 1536)


array([[ 0.09331264, -0.05714897, -0.02595772, ...,  0.00011908,
         0.00151414, -0.00072391],
       [ 0.08655477,  0.02148705, -0.09319922, ...,  0.00048628,
         0.00056532,  0.00189124]], shape=(2, 1536), dtype=float32)

In [8]:
df_nli = pd.DataFrame({
    "premise": [["el", "gato", "duerme", "en", "el", "sofa"]],
    "hypothesis": [["hay", "un", "felino", "descansando"]]
})

# pair_diff (este metodo solo se admite separando premisa y conclusion)
vectorizer = NLITextVectorizer(
    columns=("premise", "hypothesis"),
    method="pair_diff",
    word_ngram_range=(1, 1),
    max_features=5000,
    min_df=1,
    use_pos=False
)

X_vecs = vectorizer.fit_transform(df_nli)
print("Forma con pair_diff:", X_vecs.shape)
# Salida: (1, Vocabulario_Premisa + Vocabulario_Hipotesis + Vocabulario_Diferencia + 4_features_de_solapamiento)
X_vecs.toarray()

Forma con pair_diff: (1, 31)


array([[0.        , 0.38161415, 0.64612895, 0.38161415, 0.        ,
        0.38161415, 0.        , 0.38161415, 0.        , 0.5       ,
        0.        , 0.        , 0.        , 0.5       , 0.        ,
        0.5       , 0.        , 0.5       , 0.5       , 0.38161415,
        0.64612895, 0.38161415, 0.5       , 0.38161415, 0.5       ,
        0.38161415, 0.5       , 0.        , 0.        , 0.        ,
        0.        ]], dtype=float32)

In [9]:
df_svd = pd.DataFrame({
    "premise": [
        ["el", "gato", "duerme", "en", "el", "sofa"],
        ["un", "perro", "corre", "por", "el", "parque"]
    ],
    "hypothesis": [
        ["hay", "un", "felino", "descansando"],
        ["el", "animal", "esta", "quieto"]
    ]
})

vectorizer = NLITextVectorizer(
    columns=("premise", "hypothesis"),
    method="tfidf_svd",
    word_ngram_range=(1, 1),
    max_features=5000,
    min_df=1,
    svd_components=50  # Reducimos a 50 dimensiones por columna (2 columnas en total)
                       # Evito vectores enormes como tfidf
)

X_vecs = vectorizer.fit_transform(df_svd)

print("Tipo de salida:", type(X_vecs))  # numpy.ndarray (densa)
print("Forma con tfidf_svd:", X_vecs.shape)

X_vecs 

# Salida obtenida: (2, 6)

# Debería salir idealmente: (2, 100)  (50 dimensiones del vector premisa + 50 del vector hipótesis) 
# El motivo por el cual sale (2, 6) es debido al tamaño reducido de nuestro DataFrame de prueba:
# max_valid = min(matrix.shape[0] - 1, matrix.shape[1] - 1)
# 
# 1. El tamaño del corpus (matrix.shape[0]) es 4: 
#    - Se concatenan verticalmente las 2 filas de "premise" y las 2 de "hypothesis" en el fit().
#    - Por tanto, el número de documentos es 4, lo que nos da (4 - 1) = 3 como límite por filas.
#
# 2. El tamaño del vocabulario (matrix.shape[1]) es el número de palabras únicas del corpus global.
#    - Si el vocabulario global tuviera, por ejemplo, 16 palabras únicas, el límite por columnas sería (16 - 1) = 15.
#
# 3. Al hacer el min() global: max_valid = min(3 (por filas), 15 (por vocabulario)) = 3.
#
# 4. Finalmente, el SVD calcula: min(self.svd_components, max_valid) -> min(50, 3) = 3.
#    - Como el resultado es 3, el SVD solo puede generar un máximo de 3 componentes válidos por columna.
#    - Al tener 2 columnas ("premise" e "hypothesis"), se concatenan horizontalmente: 3 + 3 = 6 columnas en total.
#    - Como el DataFrame tiene 2 filas (instancias), la forma final resultante es (2, 6).

Tipo de salida: <class 'numpy.ndarray'>
Forma con tfidf_svd: (2, 6)


array([[ 0.6370161 , -0.3733249 , -0.4556326 ,  0.33051196,  0.7880738 ,
         0.2879901 ],
       [ 0.6596611 ,  0.34528226, -0.30540177,  0.58354294, -0.4291427 ,
         0.67950964]], dtype=float32)

In [10]:
df_svd = pd.DataFrame({
    "tokens": [
        ["el", "gato", "duerme", "en", "el", "sofa", "hay", "un", "felino", "descansando"],
         ["un", "perro", "corre", "por", "el", "parque", "el", "animal", "esta", "quieto"]
    ]
})

vectorizer = NLITextVectorizer(
    columns=("tokens", ),
    method="tfidf_svd",
    word_ngram_range=(1, 1),
    max_features=5000,
    min_df=1,
    svd_components=50  # Reducimos a 50 dimensiones por columna (en este caso 1 columna)
                       # Evito vectores enormes como tfidf
)

X_vecs = vectorizer.fit_transform(df_svd)

print("Tipo de salida:", type(X_vecs))  # numpy.ndarray (densa)
print("Forma con tfidf_svd:", X_vecs.shape) 

# Salida: (2, 1)

# No sale (2, 50) (ver explicacion de la celda anterior)
X_vecs

Tipo de salida: <class 'numpy.ndarray'>
Forma con tfidf_svd: (2, 1)


array([[0.7805552 ],
       [0.78055555]], dtype=float32)